# Dynamic Quantization Mechanics

(Dynamic GGUF, Selective Layer Preservation, and Dynamic NVFP4)

Once fine-tuning is finished, standard quantization tools apply a single, uniform bit-width (for example, forcing all layers strictly to INT4 or Q4_K_M).

However, neural networks do not distribute information density uniformly across layers:

- The first and last layers (input embeddings, early attention, and LM head) are hypersensitive to numerical errors; aggressive quantization here causes catastrophic perplexity spikes.
- Middle MLP / Mixture-of-Experts (MoE) layers exhibit high redundancy and can easily withstand aggressive 3-bit or 4-bit compression without accuracy loss.

Dynamic Quantization replaces uniform quantization with per-layer, mixed-precision quantization calibrated against activation divergence.

## 1. Dynamic GGUF and dynamic 2.0 allocation

When exporting fine-tuned models to local runtimes such as llama.cpp or Ollama, standard GGUF quantizers (like naive Q4_0 or Q4_K_M) quantize every tensor to the exact same format.

### Dynamic GGUF mechanism

Dynamic Quantization measures the Kullback–Leibler (KL) divergence of each layer's activations against the unquantized FP16 baseline across a small calibration set:

$$
D_{\text{KL}}(P_{\text{FP16}} \parallel Q_{\text{Quantized}}) = \sum_x P(x) \log\left(\frac{P(x)}{Q(x)}\right)
$$

```text
Unsloth Dynamic Layer Assignment
┌────────────────────────────────────────────────────────────────────────────┐
│ Layer 0-2 (Early Attention/Input)   ──► Preserved in 8-bit (Q8_0 / FP8)   │
│ Layer 3-28 (Internal MLPs / MoE)   ──► Quantized to 4-bit / 3-bit (Q4_K_M) │
│ Final Layer & LM Head (Vocab Map)  ──► Preserved in 8-bit / 16-bit (Q8_0)  │
└────────────────────────────────────────────────────────────────────────────┘
```

### Outcome

This matches the benchmark accuracy (MMLU, HumanEval) of a full 8-bit model while having nearly the disk footprint and VRAM requirement of a 4-bit model.

## How to decide per-layer precision

Not all layers degrade equally when quantized:

```text
[Input Embeddings] ──► [Early Attn (L0-L2)] ──► [Mid MLPs (L3-L20)] ──► [Late Attn (L21-L23)] ──► [LM Head]
       ▲                         ▲                      ▲                       ▲                   ▲
       │                         │                      │                       │                   │
  Keep 8/16-bit             Keep 8-bit             Aggressive 4-bit        Keep 8-bit          Keep 8/16-bit
  (High Divergence)     (Spatial Alignment)     (High Redundancy)      (Final Routing)      (Logit Collapse)
```

## Layer-wise precision pattern

```text
[Input Embeddings] ──► [Early Attn (L0-L2)] ──► [Mid MLPs (L3-L20)] ──► [Late Attn (L21-L23)] ──► [LM Head]
       ▲                         ▲                      ▲                       ▲                   ▲
       │                         │                      │                       │                   │
  Keep 8/16-bit             Keep 8-bit             Aggressive 4-bit        Keep 8-bit          Keep 8/16-bit
  (High Divergence)      (Spatial Alignment)      (High Redundancy)      (Final Routing)      (Logit Collapse)
```

This pattern reflects the fact that early and final layers are often more sensitive to quantization, while middle layers are usually more redundant.

## Decision metrics for precision allocation

### 1. Hessian trace / second-order gradients

Layers with large eigenvalues in their Hessian matrix,

$$
\frac{\partial^2 \mathcal{L}}{\partial W^2}
$$

have sharp loss landscapes.

- High curvature → keep in FP8 or 16-bit
- Flat curvature → compress to 4-bit or 3-bit

### 2. Activation outlier magnitude ($Z$-score)

If a layer contains massive activation outliers ($> 6\sigma$), 4-bit integer clipping causes quantization errors.

- If $\max(|X|) > 16 \cdot \text{std}(X)$, preserve in FP8 (E4M3) or use sub-channel scaling

### 3. Per-layer KL divergence metric

Run a forward pass on 100 calibration tokens and compare the logits before and after layer-level quantization:

$$
D_{\text{KL}}(P_{\text{FP16}} \parallel Q_{\text{Quantized}}) = \sum P \log\left(\frac{P}{Q}\right)
$$

If $D_{\text{KL}} > 0.05$, restore that specific layer to 8-bit.

## 2. Blackwell-native dynamic NVFP4 (W4A4 execution)

On modern NVIDIA Blackwell hardware architectures (RTX 50-series, B200/B300), 4-bit floating-point Tensor Cores (NVFP4) introduce a fundamental paradigm shift.

### Legacy 4-bit vs. Blackwell-native 4-bit

- Legacy 4-bit (W4A16 / bitsandbytes / AWQ): only weights are stored in 4-bit ($W4$). Activations remain in 16-bit ($A16$). The GPU must constantly dequantize weights to 16-bit to multiply them with activations, keeping Tensor Cores locked in 16-bit mode.
- Blackwell Dynamic NVFP4 (W4A4): both weights ($W4$) and incoming activations ($A4$) run in native 4-bit floating-point directly inside FP4 Tensor Cores.

### Legacy W4A16 execution

```text
[4-bit Weight] ──(Dequantize in SRAM)──► [16-bit Weight] ──┐
                                                            ├─► [16-bit Tensor Core Math]
                                   [16-bit Activation] ─────┘
```

### Blackwell Dynamic NVFP4 (W4A4 execution)

```text
[4-bit FP4 Weight] ───────┐
                          ├─► [Native FP4 Tensor Core Math (179x FLOP density vs FP32)]
[4-bit FP4 Activation] ───┘
```

### Systems advantage

- Micro-block scaling (block size 16): NVFP4 uses a micro-block size of 16 weights with an FP8 (E4M3) scale factor instead of integer power-of-two scales, isolating outlier activations and preserving mathematical precision.
- Throughput: running native $W4A4$ matrix multiplications achieves $1.5\times$ to $2.5\times$ higher inference throughput compared to standard $W4A16$ engines.